In [0]:
catalog = "firstdeproject"
schema = "silver"

tables = [t.tableName for t in spark.sql(f"SHOW TABLES IN {catalog}.{schema}").collect() if not t.isTemporary]

for t in tables:
    print(f"\n=== {catalog}.{schema}.{t} ===")
    spark.table(f"{catalog}.{schema}.{t}").show(5, truncate=False)

In [0]:
# Create dim_products

from pyspark.sql import functions as F, Window

CATALOG, SILVER, GOLD = "firstdeproject", "silver", "gold"
META = ["_source_folder", "_source_file", "_ingested_at"]

def silver(t):
    return spark.table(f"{CATALOG}.{SILVER}.{t}")

def save_gold(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{GOLD}.{name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD}")


In [0]:

# Filter to Current products only (prd_end_dt IS NULL) so each product appears once.
# `PRD_JOIN_KEY` is the column that should match sales_details.sls_prd_key.

PRD_JOIN_KEY = "prd_key"   # now the real product key (e.g. FR-R92B-58), matches sales_details.sls_prd_key

cat = silver("px_cat_g1v2").select(
    F.col("ID").alias("cat_id"),
    F.col("CAT").alias("category"),
    F.col("SUBCAT").alias("subcategory"),
    F.col("MAINTENANCE").alias("maintenance"),
)

pi = silver("prd_info").filter("prd_end_dt IS NULL")

dim_products = (
    pi.join(cat, "cat_id", "left")      # join on the shared column name, so only one cat_id is kept
      .select(
          F.row_number().over(Window.orderBy("prd_start_dt", "prd_id")).alias("product_key"),  # surrogate
          F.col("prd_id").alias("product_id"),
          F.col(PRD_JOIN_KEY).alias("product_number"),
          F.col("prd_nm").alias("product_name"),
          F.col("cat_id"),
          F.col("category"),
          F.col("subcategory"),
          F.col("maintenance"),
          F.coalesce("prd_cost", F.lit(0)).alias("cost"),
          F.when(F.upper("prd_line") == "R", "Road")
           .when(F.upper("prd_line") == "M", "Mountain")
           .when(F.upper("prd_line") == "S", "Other Sales")
           .when(F.upper("prd_line") == "T", "Touring")
           .otherwise("n/a").alias("product_line"),
          F.col("prd_start_dt").alias("start_date"),
      ))

save_gold(dim_products, "dim_products")



In [0]:
dim_products.show(5)
